# Leakage-safe feature engineering

This notebook creates current-state and past-only features for predicting `slowdown_in_5min`. The 10-minute label is retained only for later comparison.

### 1. Define paths and fingerprint protected datasets

**What the cell does:** Imports libraries, defines repository-relative paths, and records SHA-256 checksums for the labeled, segmented, and cleaned datasets.  
**Why it is needed:** Feature engineering must be reproducible and must never alter source datasets.  
**What to understand:** The displayed fingerprints identify the exact protected files and will be checked again after output creation.

In [1]:
from pathlib import Path
import hashlib

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
LABELED_INPUT_PATH = PROJECT_ROOT / "data" / "interim" / "labeled_metrics.csv"
SEGMENTED_PROTECTED_PATH = PROJECT_ROOT / "data" / "interim" / "segmented_metrics.csv"
CLEANED_PROTECTED_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned_metrics.csv"
FEATURE_OUTPUT_PATH = PROJECT_ROOT / "data" / "interim" / "feature_dataset.csv"
FEATURE_SUMMARY_PATH = PROJECT_ROOT / "reports" / "feature_engineering_summary.csv"
MISSINGNESS_OUTPUT_PATH = PROJECT_ROOT / "reports" / "feature_missingness_summary.csv"

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with path.open("rb") as file_handle:
        for chunk in iter(lambda: file_handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()

protected_paths = [LABELED_INPUT_PATH, SEGMENTED_PROTECTED_PATH, CLEANED_PROTECTED_PATH]
missing_paths = [str(path) for path in protected_paths if not path.is_file()]
if missing_paths:
    raise FileNotFoundError(f"Required datasets are missing: {missing_paths}")

hashes_before = {path: sha256_file(path) for path in protected_paths}
for path, fingerprint in hashes_before.items():
    print(f"{path}: {fingerprint}")

/Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/data/interim/labeled_metrics.csv: 43119928a5918c38b47bc1f3d7463df1e9ac0962fa591b71b99d0e710fa08cff
/Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/data/interim/segmented_metrics.csv: eafbd4f01ddf3f1b7bd5d0df11ce4b4e467c679e7718539ddbdf39b4ce6a8578
/Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/data/processed/cleaned_metrics.csv: 801976ddbe91a7415ce84038ff7df5b89c601e2201e56eaf5503154f93c2e577


### 2. Load, validate, and sort an analysis copy

**What the cell does:** Loads the labeled dataset, verifies identity and target columns, parses timestamps, checks duplicates, and stably sorts a deep copy by machine/run/segment/time.  
**Why it is needed:** Historical features require valid chronological order and unambiguous timestamps within each independent sequence.  
**What to understand:** The output summarizes machines, runs, segments, timestamp quality, and the main-target distribution before feature creation.

In [2]:
labeled_input = pd.read_csv(LABELED_INPUT_PATH)
required_columns = {
    "machine_id", "run_id", "segment_id", "timestamp", "slowdown_in_5min",
    "slowdown_in_10min", "valid_5min_horizon", "valid_10min_horizon"
}
missing_required = required_columns - set(labeled_input.columns)
if missing_required:
    raise KeyError(f"Required columns are missing: {sorted(missing_required)}")

feature_work = labeled_input.copy(deep=True)
feature_work["_source_row"] = np.arange(len(feature_work))
feature_work["_timestamp_dt"] = pd.to_datetime(feature_work["timestamp"], errors="coerce", utc=True)
invalid_timestamp_count = int(feature_work["_timestamp_dt"].isna().sum())
duplicate_timestamp_count = int(
    feature_work.duplicated(["machine_id", "run_id", "segment_id", "_timestamp_dt"], keep=False).sum()
)
if invalid_timestamp_count:
    raise ValueError(f"Invalid timestamps prevent safe rolling windows: {invalid_timestamp_count}")
if duplicate_timestamp_count:
    raise ValueError(f"Duplicated within-segment timestamps make causal ordering ambiguous: {duplicate_timestamp_count}")

sequence_keys = ["machine_id", "run_id", "segment_id"]
feature_work = feature_work.sort_values(
    sequence_keys + ["_timestamp_dt", "_source_row"], kind="mergesort"
).reset_index(drop=True)

print(f"Rows: {len(feature_work):,}")
print(f"Machines: {feature_work['machine_id'].nunique()}")
print(f"Runs: {feature_work['run_id'].nunique()}")
print(f"Segments: {feature_work['segment_id'].nunique()}")
print(f"Invalid timestamps: {invalid_timestamp_count}")
print(f"Duplicated within-segment timestamps: {duplicate_timestamp_count}")
print("5-minute label distribution, including unavailable horizons:")
display(feature_work["slowdown_in_5min"].value_counts(dropna=False).to_frame("row_count"))

Rows: 31,059
Machines: 3
Runs: 10
Segments: 367
Invalid timestamps: 0
Duplicated within-segment timestamps: 0
5-minute label distribution, including unavailable horizons:


,row_count
slowdown_in_5min,
0.0,22393
NaN,5023
1.0,3643


### 3. Select permitted raw metrics and declare exclusions

**What the cell does:** Defines the 20 allowed current metrics, 12 dynamic metrics, three historical windows, and all prohibited model inputs.  
**Why it is needed:** An explicit allow-list prevents targets, identifiers, timing outcomes, Rule C intermediates, and unavailable temperature/GPU sensors from entering the model.  
**What to understand:** Only the displayed raw metrics can become current-state model features; identifiers and labels remain non-feature columns.

In [3]:
requested_raw_features = [
    "cpu_pct", "cpu_frequency_mhz", "ram_pct", "ram_used_mb", "ram_available_mb",
    "swap_pct", "swap_used_mb", "disk_usage_pct", "disk_free_gb", "disk_read_mb_s",
    "disk_write_mb_s", "disk_latency_ms", "net_sent_mb_s", "net_recv_mb_s",
    "network_latency_ms", "process_count", "thread_count", "context_switches_per_s",
    "battery_pct", "battery_plugged"
]
raw_candidate_features = [column for column in requested_raw_features if column in feature_work.columns]
missing_requested_features = sorted(set(requested_raw_features) - set(raw_candidate_features))

dynamic_metrics = [
    "cpu_pct", "ram_pct", "swap_pct", "disk_latency_ms", "disk_read_mb_s",
    "disk_write_mb_s", "net_sent_mb_s", "net_recv_mb_s", "network_latency_ms",
    "context_switches_per_s", "process_count", "thread_count"
]
rolling_windows_seconds = [30, 60, 120]
rolling_statistics = ["mean", "max", "min", "std", "change", "missing_pct"]
difference_metrics = ["cpu_pct", "ram_pct", "swap_pct", "disk_latency_ms", "context_switches_per_s"]
missing_indicator_metrics = ["context_switches_per_s", "network_latency_ms"]

prohibited_model_columns = {
    "slowdown_now", "slowdown_in_5min", "slowdown_in_10min", "valid_5min_horizon",
    "valid_10min_horizon", "status", "ended_at_utc", "run_complete", "missed_deadline",
    "id", "machine_id", "run_id", "segment_id", "timestamp", "elapsed_seconds", "phase",
    "legacy_id", "stress_cpu_target_pct", "stress_memory_target_mb", "temperature_c",
    "gpu_usage_pct", "cpu_per_core_json", "gpu_per_device_json", "sensor_errors_json",
    "sample_reliable"
}

assert not (set(raw_candidate_features) & prohibited_model_columns)
assert set(dynamic_metrics) <= set(raw_candidate_features)
print(f"Raw candidate features available: {len(raw_candidate_features)}")
display(raw_candidate_features)
print(f"Requested features unavailable: {missing_requested_features}")

Raw candidate features available: 20


['cpu_pct',
 'cpu_frequency_mhz',
 'ram_pct',
 'ram_used_mb',
 'ram_available_mb',
 'swap_pct',
 'swap_used_mb',
 'disk_usage_pct',
 'disk_free_gb',
 'disk_read_mb_s',
 'disk_write_mb_s',
 'disk_latency_ms',
 'net_sent_mb_s',
 'net_recv_mb_s',
 'network_latency_ms',
 'process_count',
 'thread_count',
 'context_switches_per_s',
 'battery_pct',
 'battery_plugged']

Requested features unavailable: []


### 4. Create time-based past and current rolling features

**What the cell does:** For each dynamic metric and 30/60/120-second window, calculates mean, maximum, minimum, sample standard deviation, last-minus-first change, and missing percentage.  
**Why it is needed:** Sustained levels, variability, direction, and data availability can describe recent system state better than a single row.  
**What to understand:** Every window is `(current time − window, current time]`, includes the current row, uses no future row, and resets inside each machine/run/segment.

In [4]:
generated_arrays = {}
row_count = len(feature_work)
for metric in dynamic_metrics:
    for seconds in rolling_windows_seconds:
        for statistic in rolling_statistics:
            generated_arrays[f"{metric}_{statistic}_{seconds}s"] = np.full(row_count, np.nan, dtype=float)

for _, segment in feature_work.groupby(sequence_keys, sort=False):
    ordered = segment.sort_values("_timestamp_dt")
    positions = ordered.index.to_numpy()
    indexed = ordered.set_index("_timestamp_dt")
    for seconds in rolling_windows_seconds:
        window = f"{seconds}s"
        total_observations = pd.Series(1.0, index=indexed.index).rolling(window, min_periods=1, closed="right").sum()
        for metric in dynamic_metrics:
            rolling = indexed[metric].rolling(window, min_periods=1, closed="right")
            generated_arrays[f"{metric}_mean_{seconds}s"][positions] = rolling.mean().to_numpy()
            generated_arrays[f"{metric}_max_{seconds}s"][positions] = rolling.max().to_numpy()
            generated_arrays[f"{metric}_min_{seconds}s"][positions] = rolling.min().to_numpy()
            generated_arrays[f"{metric}_std_{seconds}s"][positions] = rolling.std(ddof=1).to_numpy()
            generated_arrays[f"{metric}_change_{seconds}s"][positions] = rolling.apply(
                lambda values: values.iloc[-1] - values.iloc[0]
                if pd.notna(values.iloc[-1]) and pd.notna(values.iloc[0]) else np.nan,
                raw=False,
            ).to_numpy()
            nonmissing_observations = rolling.count()
            generated_arrays[f"{metric}_missing_pct_{seconds}s"][positions] = (
                (total_observations - nonmissing_observations) / total_observations * 100
            ).to_numpy()

rolling_feature_columns = list(generated_arrays)
feature_work = pd.concat([feature_work, pd.DataFrame(generated_arrays)], axis=1)
print(f"Rolling features created: {len(rolling_feature_columns)}")
display(feature_work[sequence_keys + ["timestamp"] + rolling_feature_columns[:6]].head())

Rolling features created: 216


,machine_id,run_id,segment_id,timestamp,cpu_pct_mean_30s,cpu_pct_max_30s,cpu_pct_min_30s,cpu_pct_std_30s,cpu_pct_change_30s,cpu_pct_missing_pct_30s
0,0890dcc046c079acc4de4202,6835f125-a038-4092-beff-5107ae998b39,0890dcc046c079acc4de4202__6835f125-a038-4092-b...,2026-07-25T14:45:59.803000Z,0.000000,0.0,0.0,NaN,0.0,0.0
1,0890dcc046c079acc4de4202,6835f125-a038-4092-beff-5107ae998b39,0890dcc046c079acc4de4202__6835f125-a038-4092-b...,2026-07-25T14:46:00.857000Z,4.900000,9.8,0.0,6.929646,9.8,0.0
2,0890dcc046c079acc4de4202,6835f125-a038-4092-beff-5107ae998b39,0890dcc046c079acc4de4202__6835f125-a038-4092-b...,2026-07-25T14:46:02.829000Z,5.066667,9.8,0.0,4.908496,5.4,0.0
3,0890dcc046c079acc4de4202,6835f125-a038-4092-beff-5107ae998b39,0890dcc046c079acc4de4202__6835f125-a038-4092-b...,2026-07-25T14:46:04.845000Z,4.950000,9.8,0.0,4.014557,4.6,0.0
4,0890dcc046c079acc4de4202,6835f125-a038-4092-beff-5107ae998b39,0890dcc046c079acc4de4202__6835f125-a038-4092-b...,2026-07-25T14:46:06.853000Z,5.420000,9.8,0.0,3.632079,7.3,0.0


### 5. Add one-step differences and missingness indicators

**What the cell does:** Creates five one-step differences within each segment and two indicators for important missing sensors. It also identifies rows with less than 120 seconds of available segment history.  
**Why it is needed:** Differences capture immediate direction, indicators preserve missingness information without imputation, and the history flag quantifies limited rolling context.  
**What to understand:** Every first segment row has a missing difference; no missing value is replaced with zero.

In [5]:
difference_feature_columns = []
for metric in difference_metrics:
    feature_name = f"{metric}_diff"
    feature_work[feature_name] = feature_work.groupby(sequence_keys, sort=False)[metric].diff()
    difference_feature_columns.append(feature_name)

missing_indicator_columns = []
for metric in missing_indicator_metrics:
    feature_name = f"{metric}_missing"
    feature_work[feature_name] = feature_work[metric].isna().astype("int8")
    missing_indicator_columns.append(feature_name)

segment_start_time = feature_work.groupby(sequence_keys, sort=False)["_timestamp_dt"].transform("min")
feature_work["_history_seconds_available"] = (
    feature_work["_timestamp_dt"] - segment_start_time
).dt.total_seconds()
feature_work["_insufficient_120s_history"] = feature_work["_history_seconds_available"].lt(120)

first_row_mask = ~feature_work.duplicated(sequence_keys, keep="first")
assert feature_work.loc[first_row_mask, difference_feature_columns].isna().all().all()
assert not feature_work[missing_indicator_columns].isna().any().any()

generated_feature_columns = rolling_feature_columns + difference_feature_columns + missing_indicator_columns
print(f"Difference features: {len(difference_feature_columns)}")
print(f"Missing indicators: {len(missing_indicator_columns)}")
print(f"Rows with less than 120 seconds of segment history: {feature_work['_insufficient_120s_history'].sum():,}")

Difference features: 5
Missing indicators: 2
Rows with less than 120 seconds of segment history: 2,355


### 6. Assert temporal boundaries and absence of leakage

**What the cell does:** Verifies feature names, segment resets, first-row rolling behavior, and sampled rolling means recomputed from only timestamps at or before the current row.  
**Why it is needed:** Explicit tests guard against target leakage, future observation use, and accidental rolling across machine/run/segment boundaries.  
**What to understand:** Passing assertions mean targets, Rule C outputs, identifiers, and future rows are absent from the model-feature list.

In [6]:
model_feature_columns = raw_candidate_features + generated_feature_columns
rule_c_name_fragments = ("moderate_", "severe_", "rule_a_", "rule_b_", "rule_c_", "slowdown_now")

assert len(model_feature_columns) == len(set(model_feature_columns))
assert not (set(model_feature_columns) & prohibited_model_columns)
assert "slowdown_in_5min" not in model_feature_columns
assert "slowdown_in_10min" not in model_feature_columns
assert not any(column.startswith(rule_c_name_fragments) or column == "slowdown_now" for column in model_feature_columns)
assert "temperature_c" not in model_feature_columns and "gpu_usage_pct" not in model_feature_columns

# A segment's first rolling mean must equal its current value (or both must be missing), proving reset.
for metric in dynamic_metrics:
    for seconds in rolling_windows_seconds:
        assert np.allclose(
            feature_work.loc[first_row_mask, f"{metric}_mean_{seconds}s"].to_numpy(dtype=float),
            feature_work.loc[first_row_mask, metric].to_numpy(dtype=float),
            equal_nan=True,
        )

# Recalculate sampled means from the same sequence using only (time-window, current time].
sample_positions = np.unique(np.linspace(0, len(feature_work) - 1, 30, dtype=int))
for position in sample_positions:
    row = feature_work.iloc[position]
    for metric in dynamic_metrics:
        for seconds in rolling_windows_seconds:
            start_time = row["_timestamp_dt"] - pd.Timedelta(seconds=seconds)
            expected_window = feature_work.loc[
                feature_work["machine_id"].eq(row["machine_id"])
                & feature_work["run_id"].eq(row["run_id"])
                & feature_work["segment_id"].eq(row["segment_id"])
                & feature_work["_timestamp_dt"].gt(start_time)
                & feature_work["_timestamp_dt"].le(row["_timestamp_dt"]),
                metric,
            ]
            expected_mean = expected_window.mean()
            actual_mean = row[f"{metric}_mean_{seconds}s"]
            assert np.isclose(expected_mean, actual_mean, equal_nan=True)

leakage_risk_detected = False
print(f"Model features validated: {len(model_feature_columns)}")
print("Past-only and boundary-reset assertions passed.")
print(f"Data-leakage risk detected: {leakage_risk_detected}")

Model features validated: 243
Past-only and boundary-reset assertions passed.
Data-leakage risk detected: False


### 7. Select rows with valid 5-minute targets

**What the cell does:** Keeps only rows with a complete 5-minute horizon and a non-missing main target, while retaining raw and generated missing values.  
**Why it is needed:** Incomplete horizons cannot be training examples, but missing predictor history must be handled later inside a leakage-safe modeling pipeline.  
**What to understand:** No class balancing, imputation, normalization, splitting, or feature selection is performed.

In [7]:
modeling_row_mask = feature_work["valid_5min_horizon"].eq(True) & feature_work["slowdown_in_5min"].notna()
identification_columns = ["machine_id", "run_id", "segment_id", "timestamp"]
target_metadata_columns = [
    "valid_5min_horizon", "slowdown_in_5min", "valid_10min_horizon", "slowdown_in_10min"
]
feature_dataset = feature_work.loc[
    modeling_row_mask,
    identification_columns + model_feature_columns + target_metadata_columns,
].copy()

assert len(feature_dataset) == int(feature_work["valid_5min_horizon"].sum())
assert feature_dataset["slowdown_in_5min"].notna().all()
assert feature_dataset["valid_5min_horizon"].all()
assert len(feature_dataset) + int((~modeling_row_mask).sum()) == len(feature_work)

print(f"Input rows: {len(feature_work):,}")
print(f"Modeling rows retained: {len(feature_dataset):,}")
print("5-minute target distribution:")
display(feature_dataset["slowdown_in_5min"].value_counts().sort_index().to_frame("row_count"))

Input rows: 31,059
Modeling rows retained: 26,036
5-minute target distribution:


,row_count
slowdown_in_5min,
0.0,22393
1.0,3643


### 8. Build engineering and missingness summaries

**What the cell does:** Creates a one-row engineering summary and a long-form missingness report for every model feature overall and within each machine.  
**Why it is needed:** Feature volume, target balance, limited history, and machine-specific missingness must be known before chronological splitting and pipeline design.  
**What to understand:** High missingness remains visible and untouched; `scope=overall` summarizes the full modeling table and `scope=machine` exposes machine differences.

In [8]:
positive_target_count = int(feature_dataset["slowdown_in_5min"].eq(1).sum())
negative_target_count = int(feature_dataset["slowdown_in_5min"].eq(0).sum())
insufficient_history_output_count = int(
    feature_work.loc[modeling_row_mask, "_insufficient_120s_history"].sum()
)

feature_engineering_summary = pd.DataFrame([{
    "total_input_rows": int(len(feature_work)),
    "total_output_rows": int(len(feature_dataset)),
    "raw_candidate_feature_count": int(len(raw_candidate_features)),
    "generated_feature_count": int(len(generated_feature_columns)),
    "total_model_feature_count": int(len(model_feature_columns)),
    "valid_5min_label_count": int(modeling_row_mask.sum()),
    "positive_target_count": positive_target_count,
    "negative_target_count": negative_target_count,
    "positive_target_percentage": round(positive_target_count / len(feature_dataset) * 100, 4),
    "machine_count": int(feature_dataset["machine_id"].nunique()),
    "run_count": int(feature_dataset["run_id"].nunique()),
    "segment_count": int(feature_dataset["segment_id"].nunique()),
    "rows_with_insufficient_120s_history_input": int(feature_work["_insufficient_120s_history"].sum()),
    "rows_with_insufficient_120s_history_output": insufficient_history_output_count,
    "data_leakage_risk_detected": leakage_risk_detected,
}])

def missingness_records(dataframe, scope, machine_id=""):
    records = []
    for feature_name in model_feature_columns:
        missing_count = int(dataframe[feature_name].isna().sum())
        records.append({
            "scope": scope,
            "machine_id": machine_id,
            "feature_name": feature_name,
            "dtype": str(dataframe[feature_name].dtype),
            "missing_count": missing_count,
            "missing_percentage": round(missing_count / len(dataframe) * 100, 4) if len(dataframe) else np.nan,
        })
    return records

missingness_rows = missingness_records(feature_dataset, "overall")
for machine_id, machine_data in feature_dataset.groupby("machine_id"):
    missingness_rows.extend(missingness_records(machine_data, "machine", machine_id))
feature_missingness_summary = pd.DataFrame(missingness_rows)

high_missingness = feature_missingness_summary.loc[
    feature_missingness_summary["scope"].eq("overall")
    & feature_missingness_summary["missing_percentage"].ge(20)
].sort_values("missing_percentage", ascending=False)

display(feature_engineering_summary)
print("Overall features with at least 20% missingness:")
display(high_missingness)

,total_input_rows,total_output_rows,raw_candidate_feature_count,generated_feature_count,total_model_feature_count,valid_5min_label_count,positive_target_count,negative_target_count,positive_target_percentage,machine_count,run_count,segment_count,rows_with_insufficient_120s_history_input,rows_with_insufficient_120s_history_output,data_leakage_risk_detected
0,31059,26036,20,223,243,26036,3643,22393,13.9922,3,10,33,2355,1351,False


Overall features with at least 20% missingness:


,scope,machine_id,feature_name,dtype,missing_count,missing_percentage
240,overall,,context_switches_per_s_diff,float64,10374,39.8448
198,overall,,context_switches_per_s_change_120s,float64,10261,39.4108
192,overall,,context_switches_per_s_change_60s,float64,10013,38.4583
186,overall,,context_switches_per_s_change_30s,float64,9956,38.2394
17,overall,,context_switches_per_s,float64,7134,27.4005
180,overall,,network_latency_ms_change_120s,float64,6298,24.1896
174,overall,,network_latency_ms_change_60s,float64,6184,23.7517
168,overall,,network_latency_ms_change_30s,float64,6121,23.5098


### 9. Save outputs and perform final integrity checks

**What the cell does:** Saves the modeling table and both reports, reads them back, validates their schemas/counts, and compares protected checksums.  
**Why it is needed:** Final assertions prevent row/feature loss, target leakage, accidental imputation, and source-file modification.  
**What to understand:** A successful result is ready for chronological splitting design, while imputation and scaling must still be fitted later using training data only.

In [9]:
FEATURE_OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
FEATURE_SUMMARY_PATH.parent.mkdir(parents=True, exist_ok=True)

feature_dataset.to_csv(FEATURE_OUTPUT_PATH, index=False)
feature_engineering_summary.to_csv(FEATURE_SUMMARY_PATH, index=False)
feature_missingness_summary.to_csv(MISSINGNESS_OUTPUT_PATH, index=False)

saved_features = pd.read_csv(FEATURE_OUTPUT_PATH)
saved_summary = pd.read_csv(FEATURE_SUMMARY_PATH)
saved_missingness = pd.read_csv(MISSINGNESS_OUTPUT_PATH)

assert len(saved_features) == len(feature_dataset)
assert saved_features.columns.tolist() == feature_dataset.columns.tolist()
assert len(saved_summary) == 1
assert len(saved_missingness.loc[saved_missingness["scope"].eq("overall")]) == len(model_feature_columns)
assert not (set(model_feature_columns) & prohibited_model_columns)
assert saved_features["slowdown_in_5min"].notna().all()

hashes_after = {path: sha256_file(path) for path in protected_paths}
protected_inputs_unchanged = hashes_before == hashes_after
ready_for_chronological_splitting = bool(
    protected_inputs_unchanged
    and not leakage_risk_detected
    and len(feature_dataset) > 0
    and positive_target_count > 0
    and negative_target_count > 0
    and feature_dataset["machine_id"].nunique() > 1
    and feature_dataset["run_id"].nunique() > 1
)

print("FINAL FEATURE-ENGINEERING REPORT")
print(f"Raw candidate features: {len(raw_candidate_features)}")
print(f"Generated features: {len(generated_feature_columns)}")
print(f"Total model features: {len(model_feature_columns)}")
print(f"Output rows: {len(feature_dataset):,}")
print(f"5-minute target: positive={positive_target_count:,}, negative={negative_target_count:,}, rate={positive_target_count / len(feature_dataset) * 100:.4f}%")
print(f"High-missingness features (≥20%): {high_missingness['feature_name'].tolist()}")
print(f"Data-leakage risk detected: {leakage_risk_detected}")
print(f"Ready for chronological train/validation/test splitting design: {ready_for_chronological_splitting}")
print(f"Protected input datasets remained unchanged: {protected_inputs_unchanged}")
print("No normalization, imputation, modeling, target-based selection, or correlation-based selection was performed.")
print(f"Created: {FEATURE_OUTPUT_PATH}")
print(f"Created: {FEATURE_SUMMARY_PATH}")
print(f"Created: {MISSINGNESS_OUTPUT_PATH}")

assert protected_inputs_unchanged, "A protected input dataset changed during feature engineering."


FINAL FEATURE-ENGINEERING REPORT
Raw candidate features: 20
Generated features: 223
Total model features: 243
Output rows: 26,036
5-minute target: positive=3,643, negative=22,393, rate=13.9922%
High-missingness features (≥20%): ['context_switches_per_s_diff', 'context_switches_per_s_change_120s', 'context_switches_per_s_change_60s', 'context_switches_per_s_change_30s', 'context_switches_per_s', 'network_latency_ms_change_120s', 'network_latency_ms_change_60s', 'network_latency_ms_change_30s']
Data-leakage risk detected: False
Ready for chronological train/validation/test splitting design: True
Protected input datasets remained unchanged: True
No normalization, imputation, modeling, target-based selection, or correlation-based selection was performed.
Created: /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/data/interim/feature_dataset.csv
Created: /Users/fatimazahranamaoui/Desktop/ADOPTAI/AdoptAI_V1/reports/feature_engineering_summary.csv
Created: /Users/fatimazahranamaoui/Desktop